# Part 1 on Kaggle: two shards in parallel, one per T4

This notebook is built for **Save Version -> Save & Run All (Commit)**: Kaggle runs it
headless for up to 12 h, no browser needed, and keeps `/kaggle/working` as the output.

**One-time settings** (right-hand panel, *Session options*):
1. **Accelerator: GPU T4 x2**. Not P100: current PyTorch builds no longer support it.
2. **Internet: On** (needs a phone-verified Kaggle account).
3. **Add-ons -> Secrets**: add `GITHUB_TOKEN` (a GitHub token with read access to the
   private repo) and tick it for this notebook. Not needed if the repo is public.

**The plan** (26 runs in 5 shards):

| Session | Shards |
|---|---|
| Kaggle session 1 | `0/5` and `1/5` (shard 0 also runs the zero-shot baseline) |
| Colab (`notebook.ipynb`) | `2/5` |
| Kaggle session 2 | `3/5` and `4/5`: change `SHARDS` below, commit again |

Each shard is 5-6 runs, which should fit inside one 12 h session. A shard that runs out of
time stops cleanly (`--max-hours`) and lists what it did not start. To finish it, attach this
notebook's previous output via **Add Input -> Your Work**, and commit again with the same
`SHARDS`: earlier results are copied in and skipped.

When a session finishes: open the version -> **Output** -> download `results/`.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader
# Kaggle ships torch for its GPUs -- do NOT reinstall it.
# Exact pins: Colab and Kaggle must draw the identical TextVQA subset (see requirements.txt).
!pip -q install "transformers==5.23.0" "peft==0.21.3" "datasets==5.2.0" "accelerate==1.15.0" num2words
import torch, transformers, peft, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| datasets", datasets.__version__,
      "| GPUs", torch.cuda.device_count())
assert transformers.__version__ == "5.23.0" and datasets.__version__ == "5.2.0", "version pins not applied"
assert torch.cuda.device_count() == 2, "Set Accelerator to 'GPU T4 x2' in Session options"

In [ ]:
import os, sys, glob, shutil

# ---- the only line to change between Kaggle sessions -------------------------
SHARDS = ["0/5", "1/5"]          # session 2: ["3/5", "4/5"]
MAX_HOURS = 10.5                 # no new run starts after this; Kaggle stops at 12 h
# -----------------------------------------------------------------------------

try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _tok = None
URL = (f"https://{_tok}@github.com/minuraashen/LoRA-finetuning.git" if _tok
       else "https://github.com/minuraashen/LoRA-finetuning.git")

# Code and caches live in /tmp so they are NOT saved into the output (20 GB cap);
# only results/ and logs/ under /kaggle/working are kept.
REPO = "/tmp/LoRA-finetuning"
if not os.path.exists(REPO):
    !git clone -q {URL} {REPO}
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, "src"))
!git log --oneline -1

os.environ["RESULTS_DIR"] = "/kaggle/working/results"
os.environ["FIG_DIR"] = "/kaggle/working/figures"
os.environ["SUBSET_CACHE_DIR"] = "/tmp/subsets"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.makedirs(os.environ["RESULTS_DIR"], exist_ok=True)
os.makedirs("/kaggle/working/logs", exist_ok=True)

# Resume: results from an earlier version attached via Add Input -> Your Work.
prior = glob.glob("/kaggle/input/**/results/*.json", recursive=True)
for p in prior:
    dst = os.path.join(os.environ["RESULTS_DIR"], os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst)
print(f"copied {len(prior)} earlier result files")
for s in SHARDS:
    !python run_grid.py --tiers 1 2 3 4 --shard {s} --list

In [ ]:
# The eight pre-flight checks (CPU, ~5 min). Never skip: they catch silent mis-targeting.
!python tests/test_sanity.py

In [ ]:
# One process per GPU. Each sees exactly one T4 (CUDA_VISIBLE_DEVICES), so the
# effective batch stays 16 -- identical to the single-GPU Colab runs.
import subprocess, time, re
procs = []
for gpu, shard in enumerate(SHARDS):
    tag = shard.replace("/", "of")
    log = open(f"/kaggle/working/logs/shard{tag}.log", "w")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    cmd = ["python", "run_grid.py", "--tiers", "1", "2", "3", "4",
           "--shard", shard, "--max-hours", str(MAX_HOURS)]
    procs.append((shard, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env), log))
    print(f"started shard {shard} on GPU {gpu}")
    time.sleep(90)   # stagger: let the first process start the dataset downloads

t0 = time.time()
while any(p.poll() is None for _, p, _ in procs):
    time.sleep(600)
    print(f"--- {(time.time() - t0) / 3600:.1f} h")
    for shard, p, log in procs:
        log.flush()
        text = open(log.name, encoding="utf-8", errors="replace").read()
        done = re.findall(r"\[done\] (\S+)", text)
        fails = re.findall(r"\[FAIL\] (\S+)", text)
        state = "running" if p.poll() is None else f"exited {p.returncode}"
        print(f"shard {shard} ({state}): {len(done)} done {done[-1:] }, {len(fails)} failed {fails}")
for _, _, log in procs:
    log.close()

In [ ]:
# Summary: what this session produced. Download results/ from the version's Output tab.
import json
for shard, p, log in procs:
    print(f"===== shard {shard}: exit code {p.returncode}")
    lines = open(log.name, encoding="utf-8", errors="replace").read().splitlines()
    print("\n".join(l for l in lines if l.startswith(("[done]", "[FAIL]", "[WARN]", "[skip]"))
                    or "runs completed" in l or "not started" in l or l.startswith("failed:")))
res = sorted(glob.glob(os.path.join(os.environ["RESULTS_DIR"], "*.json")))
print(f"\n{len(res)} result files in {os.environ['RESULTS_DIR']}:")
for r in res:
    print("  ", os.path.basename(r))